# Demo: Single Sample Prediction — EXP_060A

Notebook trình diễn cách mô hình multimodal tốt nhất xử lý **một sample** gồm bình luận (text) và ảnh (images), rồi dự đoán 5 điểm đánh giá nhà hàng.

**Thí nghiệm:** `EXP_060A_bestsequential_full_configuration`

| Thành phần | Cấu hình |
|---|---|
| Image Backbone | Swin-B |
| Text Backbone | PhoBERT |
| Fusion | Cross-Attention |
| Loss | Log-Cosh |

> **Lưu ý:** Notebook này chỉ demo inference — không thực hiện XAI (Grad-CAM, SHAP, LIME).

---
## Section 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys
import json
import ast
import hashlib
import random
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

warnings.filterwarnings('ignore')

# ── paths ──
DRIVE_ROOT = "/content/drive/MyDrive/SE365"
PROJECT_DIR = DRIVE_ROOT  # project source code root
EXP_ID = "EXP_060A_bestsequential_full_configuration"
EXP_DIR = f"{DRIVE_ROOT}/experiments/{EXP_ID}"
DEMO_DIR = f"{EXP_DIR}/demo_single_sample"

DATA_DIR = f"{DRIVE_ROOT}/data/text"
IMAGE_DIR = f"{DRIVE_ROOT}/data/image"

os.makedirs(DEMO_DIR, exist_ok=True)

# add project root to path so we can import model classes
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# ── seed ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device     : {device}")
print(f"Experiment : {EXP_ID}")
print(f"EXP_DIR    : {EXP_DIR}")
print(f"DEMO_DIR   : {DEMO_DIR}")

---
## Section 2: Load Experiment Configuration

In [ ]:
# ── load config.yaml or config.json ──
config = None
config_yaml = os.path.join(EXP_DIR, 'config.yaml')
config_json = os.path.join(EXP_DIR, 'config.json')

if os.path.isfile(config_yaml):
    import yaml
    with open(config_yaml, 'r') as f:
        config = yaml.safe_load(f)
    print(f"Loaded config from: {config_yaml}")
elif os.path.isfile(config_json):
    with open(config_json, 'r') as f:
        config = json.load(f)
    print(f"Loaded config from: {config_json}")
else:
    print("WARNING: No config file found. Using defaults.")
    config = {}

# ── extract config values with defaults ──
TEXT_MODEL_NAME = config.get('text_model_name', 'vinai/phobert-base')
IMAGE_MODEL_NAME = config.get('image_model_name', 'swin_base_patch4_window7_224')
FUSION_TYPE = config.get('fusion_type', 'cross_attention')
LOSS_FN = config.get('loss_fn', 'logcosh')
MAX_LENGTH = config.get('max_length', 256)
MAX_IMAGES = 4

# ── detect checkpoint ──
CKPT_FUSION = os.path.join(EXP_DIR, 'best_model_train_fusion.pth')
CKPT_TEXT = os.path.join(EXP_DIR, 'best_model_train_text.pth')
CKPT_IMAGE = os.path.join(EXP_DIR, 'best_model_train_image.pth')

if os.path.isfile(CKPT_FUSION):
    CHECKPOINT_PATH = CKPT_FUSION
    INFERENCE_MODE = 'fusion'
elif os.path.isfile(CKPT_TEXT) and os.path.isfile(CKPT_IMAGE):
    CHECKPOINT_PATH = None  # will load separately
    INFERENCE_MODE = 'separate'
else:
    raise FileNotFoundError(
        f"No checkpoint found in {EXP_DIR}. "
        f"Expected best_model_train_fusion.pth or both text+image checkpoints."
    )

print(f"\n{'='*50}")
print(f"  Experiment ID    : {EXP_ID}")
print(f"  Text Backbone    : {TEXT_MODEL_NAME}")
print(f"  Image Backbone   : {IMAGE_MODEL_NAME}")
print(f"  Fusion Method    : {FUSION_TYPE}")
print(f"  Loss Function    : {LOSS_FN}")
print(f"  Max Text Length  : {MAX_LENGTH}")
print(f"  Inference Mode   : {INFERENCE_MODE}")
if CHECKPOINT_PATH:
    print(f"  Checkpoint       : {os.path.basename(CHECKPOINT_PATH)}")
print(f"{'='*50}")

---
## Section 3: Load Dataset & Select Sample

Chọn một sample đại diện từ tập validation hoặc test. Ưu tiên sample có nhiều ảnh, text đủ dài, và sai số ở mức trung bình (realistic).

In [ ]:
# ── detect which split to use ──
test_pred_path = os.path.join(EXP_DIR, 'test_predictions.csv')
val_pred_path = os.path.join(EXP_DIR, 'predictions.csv')

test_csv = os.path.join(DATA_DIR, 'test.csv')
val_csv = os.path.join(DATA_DIR, 'val.csv')

use_test = os.path.isfile(test_pred_path) and os.path.isfile(test_csv)

if use_test:
    split_csv = test_csv
    pred_csv = test_pred_path
    split_name = 'test'
else:
    split_csv = val_csv
    pred_csv = val_pred_path if os.path.isfile(val_pred_path) else None
    split_name = 'validation'

print(f"Using split  : {split_name}")
print(f"Split CSV    : {split_csv}")
print(f"Predictions  : {pred_csv if pred_csv else 'not available'}")

df_split = pd.read_csv(split_csv)
print(f"Split samples: {len(df_split)}")

In [ ]:
# ── select a representative sample ──
FACTOR_NAMES = ['food', 'price', 'atmos', 'service', 'overall']
LABEL_COLS = ['food_score', 'price_score', 'atmosphere_score', 'service_score', 'overall_satisfaction']
DISPLAY_NAMES = ['Food', 'Price', 'Atmosphere', 'Service', 'Overall']

selected_idx = None

if pred_csv and os.path.isfile(pred_csv):
    df_pred = pd.read_csv(pred_csv)
    # compute per-sample mean absolute error if columns exist
    error_cols = [c for c in df_pred.columns if c.startswith('absolute_error_')]
    if error_cols:
        df_pred['sample_mae'] = df_pred[error_cols].mean(axis=1)
        # pick a sample near the median error (realistic, not best/worst)
        median_err = df_pred['sample_mae'].median()
        df_pred['dist_to_median'] = (df_pred['sample_mae'] - median_err).abs()
        selected_idx = int(df_pred.sort_values('dist_to_median').iloc[0]['index'])
        print(f"Selected sample index {selected_idx} from predictions (near-median error)")

if selected_idx is None:
    # fallback: pick a sample with text and images
    candidates = df_split.copy()
    candidates = candidates.dropna(subset=['comment_clean', 'image_url'])
    candidates['text_len'] = candidates['comment_clean'].astype(str).str.len()
    candidates = candidates[candidates['text_len'] > 50]
    if len(candidates) > 0:
        selected_idx = candidates.sample(1, random_state=SEED).index[0]
    else:
        selected_idx = 0
    print(f"Selected sample index {selected_idx} (fallback selection)")

sample_row = df_split.iloc[selected_idx] if selected_idx < len(df_split) else df_split.iloc[0]
print(f"\nSample selected successfully.")

---
## Section 4: Hiển thị Input Sample

Hiển thị chính xác những gì mô hình nhận được: bình luận text và ảnh review.

In [ ]:
# ── extract sample fields ──
review_text = str(sample_row['comment_clean'])

try:
    image_urls = ast.literal_eval(sample_row['image_url'])
except Exception:
    image_urls = [sample_row['image_url']]

ground_truth = []
for col in LABEL_COLS:
    ground_truth.append(float(sample_row[col]))

review_id = sample_row.get('review_id', f'idx_{selected_idx}')

print(f"Review ID   : {review_id}")
print(f"Num images  : {len(image_urls)}")
print(f"Text length : {len(review_text)} chars")
print(f"\n{'─'*60}")
print(f"REVIEW TEXT:")
print(f"{'─'*60}")
print(review_text)
print(f"{'─'*60}")
print(f"\nGROUND TRUTH:")
for name, val in zip(DISPLAY_NAMES, ground_truth):
    print(f"  {name:<15s}: {val:.2f}")

In [ ]:
# ── load and display images ──
def load_image_from_url_or_cache(url, image_dir):
    """Load image from local cache (MD5 hash) or URL fallback."""
    url_hash = hashlib.md5(url.encode('utf-8')).hexdigest()
    local_path = os.path.join(image_dir, f"{url_hash}.jpg")
    if os.path.exists(local_path):
        return Image.open(local_path).convert('RGB'), local_path
    try:
        import requests
        response = requests.get(url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        return img, url
    except Exception:
        return Image.new('RGB', (224, 224), color='black'), 'placeholder'


loaded_images = []
image_sources = []
for url in image_urls:
    img, src = load_image_from_url_or_cache(url, IMAGE_DIR)
    loaded_images.append(img)
    image_sources.append(src)

num_real_images = len(loaded_images)

# display images in a row
n = len(loaded_images)
fig, axes = plt.subplots(1, max(n, 1), figsize=(5 * n, 5))
if n == 1:
    axes = [axes]
for i, (img, ax) in enumerate(zip(loaded_images, axes)):
    ax.imshow(img)
    ax.set_title(f'Image {i+1}', fontsize=12)
    ax.axis('off')
plt.suptitle(f'Review Images (Review ID: {review_id})', fontsize=14, y=1.02)
plt.tight_layout()
grid_path = os.path.join(DEMO_DIR, 'input_images_grid.png')
plt.savefig(grid_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Saved: {grid_path}")

---
## Section 5: Load Model & Checkpoint

Sử dụng các class thực tế từ codebase: `TextModel`, `ImageModel`, `CrossAttentionFusion`.

In [ ]:
from Models.TextModel import TextModel
from Models.ImageModel import ImageModel

# ── build fusion model ──
text_model = TextModel(model_name=TEXT_MODEL_NAME)
image_model = ImageModel(model_name=IMAGE_MODEL_NAME)

if FUSION_TYPE == 'cross_attention':
    from Models.CrossAttentionFusion import CrossAttentionFusion
    model = CrossAttentionFusion(text_model=text_model, image_model=image_model)
elif FUSION_TYPE == 'gmu':
    from Models.GMUFusion import GMUFusion
    model = GMUFusion(text_model=text_model, image_model=image_model)
elif FUSION_TYPE == 'gated_cross':
    from Models.GatedCrossModalFusion import GatedCrossModalFusion
    model = GatedCrossModalFusion(text_model=text_model, image_model=image_model)
elif FUSION_TYPE == 'film':
    from Models.FiLMFusion import FiLMFusion
    model = FiLMFusion(text_model=text_model, image_model=image_model)
else:
    from Models.FusionModel import FusionModel
    model = FusionModel(text_model=text_model, image_model=image_model)

print(f"Model class: {model.__class__.__name__}")

# ── load checkpoint ──
if INFERENCE_MODE == 'fusion':
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    state_dict = ckpt['model_state_dict'] if 'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    print(f"Loaded fusion checkpoint: {os.path.basename(CHECKPOINT_PATH)}")
    if 'best_mean_mae' in ckpt:
        print(f"Checkpoint best_mean_mae: {ckpt['best_mean_mae']:.4f}")
else:
    # load text and image weights separately into the fusion model's sub-modules
    ckpt_t = torch.load(CKPT_TEXT, map_location=device)
    sd_t = ckpt_t['model_state_dict'] if 'model_state_dict' in ckpt_t else ckpt_t
    model.text_model.load_state_dict(sd_t)
    ckpt_i = torch.load(CKPT_IMAGE, map_location=device)
    sd_i = ckpt_i['model_state_dict'] if 'model_state_dict' in ckpt_i else ckpt_i
    model.image_model.load_state_dict(sd_i)
    print(f"Loaded text + image checkpoints separately")

model.to(device)
model.eval()
print(f"Model on {device}, eval mode.")

---
## Section 6: Preprocess Sample

Áp dụng chính xác pipeline preprocessing như khi huấn luyện: tokenize text, transform ảnh, padding multi-image.

In [ ]:
from transformers import AutoTokenizer, AutoImageProcessor

# ── tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)

text_inputs = tokenizer(
    review_text,
    truncation=True,
    padding='max_length',
    max_length=MAX_LENGTH,
    return_tensors='pt',
)

# ── image processor ──
try:
    image_processor = AutoImageProcessor.from_pretrained(IMAGE_MODEL_NAME)
except Exception:
    if 'siglip' in IMAGE_MODEL_NAME.lower():
        image_processor = AutoImageProcessor.from_pretrained('google/siglip-base-patch16-256')
    else:
        import timm
        class TimmProcessor:
            def __init__(self, model_name):
                data_config = timm.data.resolve_model_data_config(model_name)
                self.transform = timm.data.create_transform(**data_config, is_training=False)
            def __call__(self, images, return_tensors='pt'):
                pixel_values = torch.stack([self.transform(img.convert('RGB')) for img in images])
                return {'pixel_values': pixel_values}
        image_processor = TimmProcessor(IMAGE_MODEL_NAME)

# ── prepare images: pad to MAX_IMAGES with black images ──
images_for_model = list(loaded_images[:MAX_IMAGES])
num_images_actual = len(images_for_model)
while len(images_for_model) < MAX_IMAGES:
    images_for_model.append(Image.new('RGB', (224, 224), color='black'))

image_inputs = image_processor(images_for_model, return_tensors='pt')['pixel_values']

# ── build input tensors ──
input_ids = text_inputs['input_ids'].to(device)                    # (1, seq_len)
attention_mask = text_inputs['attention_mask'].to(device)          # (1, seq_len)
pixel_values = image_inputs.unsqueeze(0).to(device)                # (1, max_images, C, H, W)
num_images_tensor = torch.tensor([num_images_actual], dtype=torch.long).to(device)  # (1,)

print(f"input_ids      : {input_ids.shape}")
print(f"attention_mask  : {attention_mask.shape}")
print(f"pixel_values    : {pixel_values.shape}")
print(f"num_images      : {num_images_tensor} (actual: {num_images_actual}, padded to {MAX_IMAGES})")

---
## Section 7: Run Inference

In [ ]:
with torch.no_grad():
    output = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        num_images=num_images_tensor,
    )
    preds = output[0] if isinstance(output, tuple) else output

preds_np = preds.cpu().numpy().flatten()  # shape (5,)

# factor order from codebase: [food, price, atmos, service, overall]
pred_dict = {
    'Food': preds_np[0],
    'Price': preds_np[1],
    'Atmosphere': preds_np[2],
    'Service': preds_np[3],
    'Overall': preds_np[4],
}

print("Inference complete.")
for name, val in pred_dict.items():
    print(f"  {name:<15s}: {val:.4f}")

---
## Section 8: Prediction Result Table

So sánh Ground Truth vs Prediction cho từng tiêu chí.

In [ ]:
# ground_truth order matches LABEL_COLS: [food, price, atmos, service, overall]
gt_dict = {
    'Food': ground_truth[0],
    'Price': ground_truth[1],
    'Atmosphere': ground_truth[2],
    'Service': ground_truth[3],
    'Overall': ground_truth[4],
}

df_result = pd.DataFrame({
    'Target': DISPLAY_NAMES,
    'Ground Truth': [gt_dict[n] for n in DISPLAY_NAMES],
    'Prediction': [pred_dict[n] for n in DISPLAY_NAMES],
})
df_result['Absolute Error'] = (df_result['Ground Truth'] - df_result['Prediction']).abs()

# ── aggregate metrics ──
sample_mean_mae = df_result['Absolute Error'].mean()
sample_overall_error = df_result.loc[df_result['Target'] == 'Overall', 'Absolute Error'].values[0]
aspect_mask = df_result['Target'] != 'Overall'
sample_aspect_mae = df_result.loc[aspect_mask, 'Absolute Error'].mean()

print(f"{'='*55}")
print(f"  Sample Mean MAE     : {sample_mean_mae:.4f}")
print(f"  Overall Abs Error   : {sample_overall_error:.4f}")
print(f"  Aspect MAE (4 asp)  : {sample_aspect_mae:.4f}")
print(f"{'='*55}\n")

df_result.style.format({'Ground Truth': '{:.2f}', 'Prediction': '{:.4f}', 'Absolute Error': '{:.4f}'})

In [ ]:
# ── save result CSV ──
csv_path = os.path.join(DEMO_DIR, 'single_sample_result.csv')
df_result.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

---
## Section 9: Visualization

In [ ]:
# ── Figure 1: Ground Truth vs Prediction ──
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(DISPLAY_NAMES))
bar_width = 0.35

bars_gt = ax.bar(x - bar_width/2, df_result['Ground Truth'], bar_width,
                 label='Ground Truth', color='#2196F3', edgecolor='white', linewidth=0.8)
bars_pred = ax.bar(x + bar_width/2, df_result['Prediction'], bar_width,
                   label='Prediction', color='#FF5722', edgecolor='white', linewidth=0.8)

for bar in bars_gt:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, color='#1565C0')
for bar in bars_pred:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, color='#BF360C')

ax.set_xlabel('Target')
ax.set_ylabel('Score')
ax.set_title(f'Ground Truth vs Prediction — {EXP_ID}', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(DISPLAY_NAMES)
ax.legend(loc='upper right')
ax.set_ylim(0, max(df_result[['Ground Truth', 'Prediction']].max()) + 1.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig1_path = os.path.join(DEMO_DIR, 'prediction_vs_groundtruth.png')
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig1_path}")

In [ ]:
# ── Figure 2: Absolute Error per Target ──
fig, ax = plt.subplots(figsize=(8, 4))

colors = ['#4CAF50' if e < 1.0 else '#FF9800' if e < 2.0 else '#F44336'
          for e in df_result['Absolute Error']]

bars = ax.bar(DISPLAY_NAMES, df_result['Absolute Error'], color=colors,
              edgecolor='white', linewidth=0.8)

for bar, err in zip(bars, df_result['Absolute Error']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{err:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(y=sample_mean_mae, color='#9E9E9E', linestyle='--', linewidth=1,
           label=f'Mean MAE = {sample_mean_mae:.3f}')
ax.set_xlabel('Target')
ax.set_ylabel('Absolute Error')
ax.set_title(f'Absolute Error per Target — {EXP_ID}', fontsize=12)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig2_path = os.path.join(DEMO_DIR, 'absolute_error_per_target.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig2_path}")

---
## Section 10: Nhận xét Kết quả

Mô tả ngắn gọn về kết quả dự đoán trên sample này.

In [ ]:
# ── generate natural-language interpretation ──
best_target = df_result.loc[df_result['Absolute Error'].idxmin(), 'Target']
worst_target = df_result.loc[df_result['Absolute Error'].idxmax(), 'Target']
best_err = df_result['Absolute Error'].min()
worst_err = df_result['Absolute Error'].max()

interpretation = (
    f"Trên sample này (Review ID: {review_id}), mô hình {EXP_ID} "
    f"cho Mean MAE = {sample_mean_mae:.3f} trên thang 10 điểm.\n\n"
    f"- Dự đoán chính xác nhất ở tiêu chí **{best_target}** "
    f"(sai lệch chỉ {best_err:.3f} điểm).\n"
    f"- Sai số lớn nhất nằm ở tiêu chí **{worst_target}** "
    f"({worst_err:.3f} điểm), cho thấy sample này có thể khó đánh giá "
    f"ở khía cạnh đó.\n\n"
    f"Overall satisfaction được dự đoán với sai lệch {sample_overall_error:.3f} điểm "
    f"so với ground truth.\n\n"
    f"*Lưu ý: Đây là nhận xét mô tả, không phải phân tích XAI. "
    f"Phân tích Grad-CAM, SHAP, LIME sẽ được thực hiện ở Phase 8.*"
)

print(interpretation)

In [ ]:
# ── save summary markdown ──
summary_lines = [
    f"# Demo Single Sample — {EXP_ID}",
    "",
    f"**Review ID:** {review_id}",
    f"**Split:** {split_name}",
    f"**Number of images:** {num_real_images}",
    f"**Text length:** {len(review_text)} characters",
    "",
    "## Configuration",
    "",
    f"| Component | Value |",
    f"|---|---|",
    f"| Image Backbone | {IMAGE_MODEL_NAME} |",
    f"| Text Backbone | {TEXT_MODEL_NAME} |",
    f"| Fusion | {FUSION_TYPE} |",
    f"| Loss | {LOSS_FN} |",
    "",
    "## Review Text",
    "",
    f"> {review_text}",
    "",
    "## Results",
    "",
    "| Target | Ground Truth | Prediction | Absolute Error |",
    "|---|---:|---:|---:|",
]
for _, row in df_result.iterrows():
    summary_lines.append(
        f"| {row['Target']} | {row['Ground Truth']:.2f} | "
        f"{row['Prediction']:.4f} | {row['Absolute Error']:.4f} |"
    )
summary_lines += [
    "",
    f"**Sample Mean MAE:** {sample_mean_mae:.4f}",
    f"**Overall Abs Error:** {sample_overall_error:.4f}",
    f"**Aspect MAE:** {sample_aspect_mae:.4f}",
    "",
    "## Interpretation",
    "",
    interpretation,
]

summary_path = os.path.join(DEMO_DIR, 'single_sample_summary.md')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(summary_lines))
print(f"Saved: {summary_path}")

---
## Section 11: Demo Summary

In [ ]:
print(f"{'='*60}")
print(f"  DEMO COMPLETE")
print(f"{'='*60}")
print(f"  Experiment    : {EXP_ID}")
print(f"  Model         : {model.__class__.__name__}")
print(f"  Sample ID     : {review_id}")
print(f"  Split         : {split_name}")
print(f"  Num images    : {num_real_images}")
print(f"  Text length   : {len(review_text)} chars")
print(f"{'─'*60}")
print(f"  Mean MAE      : {sample_mean_mae:.4f}")
print(f"  Overall Error : {sample_overall_error:.4f}")
print(f"  Aspect MAE    : {sample_aspect_mae:.4f}")
print(f"{'─'*60}")
print(f"  Artifacts saved to:")
print(f"    {DEMO_DIR}/")
for fname in ['input_images_grid.png', 'prediction_vs_groundtruth.png',
              'absolute_error_per_target.png', 'single_sample_result.csv',
              'single_sample_summary.md']:
    fpath = os.path.join(DEMO_DIR, fname)
    status = 'OK' if os.path.isfile(fpath) else 'MISSING'
    print(f"    [{status}] {fname}")
print(f"{'='*60}")